# 🌍 VoxAtlas — omniASR Transcription Demo (Free Colab + CTC 300M)

This notebook lets you run Meta's **omniASR CTC 300M** model on Google Colab's free T4 GPU.

**What you'll do:**
1. Install the `omnilingual-asr` package and dependencies
2. Load audio samples from the HuggingFace omnilingual ASR corpus
3. Transcribe speech in any of 1,600+ languages
4. Calculate Character Error Rate (CER) against ground truth
5. Compare results across multiple languages

**Requirements:**
- Google Colab with **GPU runtime** → Runtime → Change runtime type → **T4 GPU**
- HuggingFace token (for accessing the gated audio corpus)

---

## Step 1: Check GPU & Install Dependencies

First, verify you have a GPU runtime, then install the required packages.

In [ ]:
# Verify GPU is available
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"✅ GPU available: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("❌ No GPU detected! Go to Runtime → Change runtime type → T4 GPU")
    raise SystemExit("GPU required")

In [ ]:
# Install omnilingual-asr and audio dependencies
!pip install -q omnilingual-asr datasets soundfile librosa jiwer
print("✅ All packages installed")

## Step 2: Authenticate with HuggingFace

Paste your HuggingFace token below. You can get one at https://huggingface.co/settings/tokens

In [ ]:
from huggingface_hub import login
from getpass import getpass

# This will prompt you for your token securely
HF_TOKEN = getpass("Enter your HuggingFace token: ")
login(token=HF_TOKEN)
print("✅ Authenticated with HuggingFace")

## Step 3: Load the omniASR CTC 300M Model

This is the smallest and fastest model — ideal for free-tier Colab.
First download takes a few minutes; subsequent runs use the cached weights.

In [ ]:
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline
import time

MODEL_CARD = "omniASR_CTC_300M_v2"

print(f"Loading {MODEL_CARD}...")
start = time.time()
pipeline = ASRInferencePipeline(model_card=MODEL_CARD)
print(f"✅ Model loaded in {time.time() - start:.1f}s")
print(f"   Model size: ~300M parameters")
print(f"   GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## Step 4: Load Audio Samples from HuggingFace Corpus

The `facebook/omnilingual-asr-corpus` dataset contains audio samples for 1,600+ languages.
Let's load a few samples and transcribe them.

In [ ]:
from datasets import load_dataset
import numpy as np

def load_audio_sample(lang_code, sample_idx=0):
    """Load a single audio sample from the omnilingual corpus."""
    ds = load_dataset(
        "facebook/omnilingual-asr-corpus",
        lang_code,
        split="train",
        streaming=True,
        token=HF_TOKEN,
    )
    for i, sample in enumerate(ds):
        if i == sample_idx:
            audio = sample["audio"]
            return {
                "waveform": np.array(audio["array"], dtype=np.float32),
                "sample_rate": audio["sampling_rate"],
                "ground_truth": sample.get("raw_text", ""),
                "lang_code": lang_code,
            }
    return None

# Test with English
print("Loading English sample...")
sample = load_audio_sample("eng_Latn")
if sample:
    print(f"✅ Loaded audio: {len(sample['waveform'])} samples @ {sample['sample_rate']}Hz")
    print(f"   Duration: {len(sample['waveform']) / sample['sample_rate']:.1f}s")
    print(f"   Ground truth: {sample['ground_truth'][:100]}")
else:
    print("❌ Could not load sample")

## Step 5: Transcribe a Single Sample

Run the CTC 300M model on the loaded audio and compute CER.

In [ ]:
from jiwer import cer as compute_cer

def transcribe_sample(pipeline, sample):
    """Transcribe audio and compute CER."""
    waveform = sample["waveform"]
    
    start = time.time()
    transcriptions = pipeline.transcribe([waveform], batch_size=1)
    latency_ms = int((time.time() - start) * 1000)
    
    prediction = transcriptions[0]
    ground_truth = sample["ground_truth"]
    
    # Compute CER
    if ground_truth and prediction:
        error_rate = compute_cer(ground_truth, prediction) * 100
    else:
        error_rate = -1
    
    return {
        "prediction": prediction,
        "ground_truth": ground_truth,
        "cer": round(error_rate, 2),
        "latency_ms": latency_ms,
    }

# Transcribe the English sample
result = transcribe_sample(pipeline, sample)

print(f"Language:     {sample['lang_code']}")
print(f"Ground truth: {result['ground_truth'][:100]}")
print(f"Prediction:   {result['prediction'][:100]}")
print(f"CER:          {result['cer']}%")
print(f"Latency:      {result['latency_ms']}ms")

## Step 6: Batch Transcribe Multiple Languages

Let's transcribe samples from a diverse set of languages and compare CER.

In [ ]:
# Diverse set of languages to test
TEST_LANGUAGES = [
    ("eng_Latn", "English"),
    ("fra_Latn", "French"),
    ("spa_Latn", "Spanish"),
    ("cmn_Hans", "Mandarin Chinese"),
    ("hin_Deva", "Hindi"),
    ("ara_Arab", "Arabic"),
    ("swa_Latn", "Swahili"),
    ("yor_Latn", "Yoruba"),
    ("hau_Latn", "Hausa"),
    ("tgl_Latn", "Tagalog"),
]

results = []
for lang_code, lang_name in TEST_LANGUAGES:
    print(f"\n--- {lang_name} ({lang_code}) ---")
    try:
        sample = load_audio_sample(lang_code)
        if sample is None:
            print(f"  ⚠️  No audio sample found")
            continue
        result = transcribe_sample(pipeline, sample)
        result["lang_code"] = lang_code
        result["lang_name"] = lang_name
        results.append(result)
        print(f"  Ground truth: {result['ground_truth'][:80]}")
        print(f"  Prediction:   {result['prediction'][:80]}")
        print(f"  CER: {result['cer']}%  |  Latency: {result['latency_ms']}ms")
    except Exception as e:
        print(f"  ❌ Error: {e}")

print(f"\n✅ Transcribed {len(results)} languages")

## Step 7: Visualize Results

Plot CER across languages as a bar chart.

In [ ]:
import matplotlib.pyplot as plt

if results:
    names = [r["lang_name"] for r in results]
    cers = [r["cer"] for r in results]
    colors = ["#22C55E" if c < 5 else "#84CC16" if c < 10 else "#F59E0B" if c < 20 else "#F97316" if c < 30 else "#EF4444" for c in cers]

    fig, ax = plt.subplots(figsize=(12, 5))
    bars = ax.barh(names, cers, color=colors, edgecolor="white", height=0.6)
    ax.set_xlabel("Character Error Rate (%)")
    ax.set_title(f"omniASR CTC 300M — CER by Language", fontsize=14, fontweight="bold")
    ax.invert_yaxis()
    ax.spines[["top", "right"]].set_visible(False)

    for bar, cer in zip(bars, cers):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                f"{cer:.1f}%", va="center", fontsize=10)

    plt.tight_layout()
    plt.show()
else:
    print("No results to plot")

## Step 8: Try Your Own Language

Change the `lang_code` below to any of the 1,600+ supported languages.

Language codes follow the format `iso639-3_Script` (e.g., `eng_Latn`, `cmn_Hans`, `ara_Arab`).

Full list: see the VoxAtlas Explorer page or the `per_language_results.csv` file.

In [ ]:
# ✏️ Change this to any language code
LANG_CODE = "ibo_Latn"  # Igbo

sample = load_audio_sample(LANG_CODE)
if sample:
    result = transcribe_sample(pipeline, sample)
    print(f"Language:     {LANG_CODE}")
    print(f"Ground truth: {result['ground_truth']}")
    print(f"Prediction:   {result['prediction']}")
    print(f"CER:          {result['cer']}%")
    print(f"Latency:      {result['latency_ms']}ms")
else:
    print(f"Could not load audio for {LANG_CODE}")

## Step 9: Export Results to CSV

Save your transcription results for import into VoxAtlas.

In [ ]:
import csv
import io

if results:
    output_path = "omniASR_CTC_300M_results.csv"
    with open(output_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["lang_code", "lang_name", "prediction", "ground_truth", "cer", "latency_ms"])
        writer.writeheader()
        writer.writerows(results)
    print(f"✅ Results saved to {output_path}")
    
    # Download link in Colab
    try:
        from google.colab import files
        files.download(output_path)
    except ImportError:
        pass  # Not running in Colab
else:
    print("No results to export")

---

## 📊 GPU Memory & Performance Summary

In [ ]:
print("=" * 50)
print("GPU Summary")
print("=" * 50)
print(f"Model:           {MODEL_CARD}")
print(f"GPU:             {torch.cuda.get_device_name(0)}")
print(f"Memory used:     {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Memory cached:   {torch.cuda.memory_reserved() / 1e9:.2f} GB")
print(f"Languages tested: {len(results)}")
if results:
    avg_cer = sum(r['cer'] for r in results) / len(results)
    avg_lat = sum(r['latency_ms'] for r in results) / len(results)
    print(f"Avg CER:         {avg_cer:.1f}%")
    print(f"Avg latency:     {avg_lat:.0f}ms")